[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/optics/crystal_diffraction/crystal_diffraction_space.ipynb)

# Crystal Deformation and Diffraction in the Algebra of Space

This example computes how a crystal's diffraction pattern shifts and shears when the crystal deforms, using scalar-plus-pseudoscalar phasors in 3D space as a conventional comparison twin to the spacetime electromagnetic formulation. When a crystal stretches, its atoms move apart while its diffraction peaks move closer together so that waves scattered from every atom continue to arrive in phase at each peak.

In the 3D geometric algebra of space, atom positions are intrinsic antivector points and wavevectors are extrinsic planes through the origin whose pairing `wavevector & position` measures phase without a metric. A spatial deformation maps points forward via `Point <- Point` while its adjugate pulls planes back via `deformation.adjugate()(k) & x == k & deformation(x)`, so `deformation.adjugate().solve(lattice)` carries every reciprocal peak forward directly, and exponentiating the spatial pseudoscalar in `(xyz * phases).exp()` forms a complex-like phasor whose handedness cancels out when `.scalar_norm_squared()` reads the intensity; compare the spacetime electromagnetic formulation in [crystal_diffraction.ipynb](crystal_diffraction.ipynb) and see [ORNL's diffraction primer](https://neutrons.ornl.gov/sites/default/files/NX_Miller_2018.pdf).

In matrix crystallography and scalar diffraction notation, a deformation gradient $F$ maps atom coordinate vectors by $\mathbf{x}'_j = F\mathbf{x}_j$ while reciprocal lattice vectors transform by the inverse transpose $\mathbf{g}' = F^{-\mathsf{T}}\mathbf{g}$ so that $(\mathbf{g}')^{\mathsf{T}}\mathbf{x}'_j = \mathbf{g}^{\mathsf{T}}\mathbf{x}_j$, and coherent kinematic scattering sums complex scalar exponentials into the normalized structure-factor intensity $I(\mathbf{q}) = \bigl|\tfrac{1}{N}\sum_{j=1}^N e^{i\,\mathbf{q}\cdot\mathbf{x}'_j}\bigr|^2$.

In [ ]:
# In Colab, numga and its examples, from GitHub.
import sys

if "google.colab" in sys.modules:
    %pip install -q "numga[examples] @ https://github.com/EelcoHoogendoorn/numga/archive/HEAD.tar.gz"

In [ ]:
%matplotlib inline
import numpy as np

from examples.optics.crystal_diffraction import render
from examples.util.animation import save_figure
from numga import NumpyContext
from numga.algebras import VGA3D

np.set_printoptions(precision=3, suppress=True)

ga = VGA3D
mv = NumpyContext(ga).multivector
Scalar = ga.gatype.scalar()                                                     # Scalar+
Plane = ga.gatype.vector()                                                      # Vector+
Point = ga.gatype.antivector()                                                  # Antivector-
Deformation = ga.gatype((Point, Point))                                         # Antivector- <- Antivector-
Phasor = ga.gatype.scalar_pseudoscalar()                                        # Scalar+ + Pseudoscalar+

## Lattice deformation and its adjugate

Atoms in the undeformed crystal sit on a finite square grid in the xy plane, and its reciprocal lattice consists of the origin planes whose pairing with every atom is an integer multiple of a full turn. Applying `layer_deformation` stretches, shears, and rotates the atom points, while solving `adjugates[:, None].solve(lattice)` finds the deformed reciprocal planes that preserve every atom's phase.

In [ ]:
# Unstrained square layer, uniaxial x-stretch, and combined stretch, shear, and rotation.
n_shells = 3
n_orders = 3
n_atoms = (2 * n_shells + 1) ** 2
n_reflections = (2 * n_orders + 1) ** 2
spacing = 1.0
stretches = np.array([1.0, 1.7, 1.7])
shears = np.array([0.0, 0.0, 0.6])
angles = np.array([0.0, 0.0, np.pi / 9])
n_cases = len(stretches)

In [ ]:
shell_indices = np.arange(-n_shells, n_shells + 1)
atoms = (mv.x * shell_indices[None, :] + mv.y * shell_indices[:, None]).reshape(-1).dual() * spacing  # [n_atoms] Point-
order_indices = np.arange(-n_orders, n_orders + 1)
lattice = (
    (mv.x * order_indices[None, :] + mv.y * order_indices[:, None]).reshape(-1) * (2 * np.pi / spacing)
)                                                                               # [n_reflections] Plane+


def layer_deformation(stretches: np.ndarray, shears: np.ndarray, angles: np.ndarray) -> Deformation:
    """Planar stretches along x, shears from y into x, and xy rotations form spatial point maps (`[...] Point- <- Point-`)."""
    local_map = (
        Point
        + (stretches - 1) * mv.x.dual() * (mv.x & Point)
        + shears * mv.x.dual() * (mv.y & Point)
    )                                                                           # [...] Point- <- Point-
    rotors = (mv.xy * (-angles / 2)).exp()                                      # [...] Rotor+
    return rotors >> local_map                                                  # [...] Point- <- Point-


deformations = layer_deformation(stretches, shears, angles)                     # [n_cases] Point- <- Point-
positions = deformations[:, None](atoms)                                        # [n_cases, n_atoms] Point-

# The adjugate pulls phase measurements back; solving through it carries reciprocal planes forward.
adjugates = deformations.adjugate()                                             # [n_cases] Plane+ <- Plane+
reciprocal = adjugates[:, None].solve(lattice)                                  # [n_cases, n_reflections] Plane+

Comparing the coefficient tables of the combined stretch-shear-rotation map on points and its adjugate on planes shows how duality pairs each basis plane `x`, `y`, `z` with its complementary point `yz`, `zx`, `xy` without using a metric.

In matrix notation, the adjugate's matrix in the dual basis is the transpose $F^{\mathsf{T}}$ of the deformation matrix $F$, because incidence pairing needs no metric inner product to convert planes into points.

In [ ]:
print(f"deformation\n{deformations[-1].coefficient_table()}\n")
print(f"adjugate\n{adjugates[-1].coefficient_table()}\n")

## Scattered phasor waves and intensity

At each momentum transfer, exponentiating the spatial pseudoscalar times the phase `(xyz * phases).exp()` turns a scalar-plus-pseudoscalar phasor by each atom's phase, and `.scalar_norm_squared()` reads the coherent intensity of the averaged amplitude so every reciprocal lattice peak reaches unit intensity.

In [ ]:
half_width = 10.0
n_pixels = 101
samples = np.linspace(-half_width, half_width, n_pixels)
transfers = mv.x * samples[None, :] + mv.y * samples[:, None]                   # [n_pixels, n_pixels] Plane+

In [ ]:
def scattered_intensity(positions: Point, transfers: Plane) -> Scalar:
    """Coherent scattering from equal atoms (`[..., n_atoms] Point-`) across `transfers` (`[..., n_rows, n_columns] Plane+`) sums pseudoscalar phasors into normalized intensities (`[..., n_rows, n_columns] Scalar+`)."""
    n_atoms = positions.shape[-1]
    phases = transfers[..., None] & positions[..., None, None, :]               # [..., n_atoms] Scalar+
    waves = (mv.xyz * phases).exp()                                             # [..., n_atoms] Phasor+
    amplitude = waves.sum(axis=-1) / n_atoms                                    # [...] Phasor+
    return amplitude.scalar_norm_squared()                                      # [...] Scalar+


intensities = scattered_intensity(positions, transfers)                         # [n_cases, n_pixels, n_pixels] Scalar+
save_figure(render.comparison_figure(positions, reciprocal, transfers, intensities), "crystal_diffraction_space");

Each column pairs one deformation state's atom positions in the top row with its logarithmic diffraction intensity across the momentum-transfer plane in the bottom row, where cyan rings mark the carried reciprocal lattice wavevectors.

In [ ]:
# checks
moved_phases = reciprocal[:, :, None] & positions[:, None, :]                   # [n_cases, n_reflections, n_atoms] Scalar+
rest_phases = lattice[:, None] & atoms[None, :]                                 # [n_reflections, n_atoms] Scalar+
pullback_phases = adjugates[:, None](lattice)[:, :, None] & atoms               # [n_cases, n_reflections, n_atoms] Scalar+
pushed_phases = lattice[:, None] & positions[:, None, :]                        # [n_cases, n_reflections, n_atoms] Scalar+
assert ((moved_phases - rest_phases).abs() < 1e-6).all()
assert ((pullback_phases - pushed_phases).abs() < 1e-6).all()

peaks = scattered_intensity(positions, reciprocal[:, None, :])                  # [n_cases, 1, n_reflections] Scalar+
assert ((peaks - 1).abs() < 1e-5).all()
assert (intensities >= 0).all()